In [43]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [44]:
# ============================================================
# PART 1 : INSTALLATION + IMPORTS + CONFIGURATION
# FLEURS Spanish ASR
# Wav2Vec2-XLS-R
# Transformers 5.x Compatible
# ============================================================

# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================

!pip -q install -U transformers
!pip -q install -U datasets
!pip -q install -U evaluate
!pip -q install -U jiwer
!pip -q install -U accelerate
!pip -q install -U librosa
!pip -q install -U soundfile
!pip -q install -U sentencepiece

# ============================================================
# IMPORTS
# ============================================================

import os
import re
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd

import torch
import torchaudio

from datasets import load_dataset, Audio

import evaluate

from transformers import (
    AutoProcessor,
    AutoModelForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

warnings.filterwarnings("ignore")

# ============================================================
# RANDOM SEED
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("DEVICE :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 70)

# ============================================================
# MODEL
# ============================================================

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

print("Model :", MODEL_NAME)

# ============================================================
# DATASET
# ============================================================

DATASET_NAME = "google/fleurs"
LANGUAGE = "es_419"

print("Dataset :", DATASET_NAME)
print("Language :", LANGUAGE)

print("=" * 70)

# ============================================================
# TRAINING CONFIGURATION
# ============================================================

SAMPLING_RATE = 16000

NUM_EPOCHS = 20

LEARNING_RATE = 1e-4

TRAIN_BATCH_SIZE = 8

EVAL_BATCH_SIZE = 8

GRADIENT_ACCUMULATION = 2

WARMUP_RATIO = 0.10

WEIGHT_DECAY = 0.005

print("Training Configuration")
print("-" * 70)
print("Epochs              :", NUM_EPOCHS)
print("Learning Rate       :", LEARNING_RATE)
print("Train Batch Size    :", TRAIN_BATCH_SIZE)
print("Eval Batch Size     :", EVAL_BATCH_SIZE)
print("Gradient Accum      :", GRADIENT_ACCUMULATION)
print("Sampling Rate       :", SAMPLING_RATE)

print("=" * 70)

# ============================================================
# LOAD METRICS
# ============================================================

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print("Metrics Loaded Successfully")

print("=" * 70)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Environment Ready")

print("=" * 70)

DEVICE : cuda
GPU : Tesla T4
Model : facebook/wav2vec2-xls-r-300m
Dataset : google/fleurs
Language : es_419
Training Configuration
----------------------------------------------------------------------
Epochs              : 20
Learning Rate       : 0.0001
Train Batch Size    : 8
Eval Batch Size     : 8
Gradient Accum      : 2
Sampling Rate       : 16000
Metrics Loaded Successfully
Environment Ready


In [45]:
# ============================================================
# PART 2 : LOAD FLEURS SPANISH DATASET
# ============================================================

print("=" * 70)
print("LOADING FLEURS SPANISH DATASET")
print("=" * 70)

dataset = load_dataset(
    DATASET_NAME,
    LANGUAGE
)

print(dataset)

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("Train Samples      :", len(dataset["train"]))
print("Validation Samples :", len(dataset["validation"]))
print("Test Samples       :", len(dataset["test"]))

print("\nDataset Features:")
print(dataset["train"].column_names)

# ============================================================
# VERIFY SAMPLE
# ============================================================

print("\n" + "=" * 70)
print("VERIFY SPANISH DATASET")
print("=" * 70)

sample = dataset["train"][0]

print("Audio Path:")
print(sample["path"])

print("\nNumber of Samples:")
print(sample["num_samples"])

print("\nLanguage:")
print(sample["language"])

print("\nGender:")
print(sample["gender"])

print("\nSpanish Transcription:")
print(sample["transcription"])

print("=" * 70)

# ============================================================
# AUDIO CASTING
# ============================================================

dataset = dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

print("\nAudio successfully resampled to 16 kHz")

print("=" * 70)

# ============================================================
# CHECK AUDIO INFORMATION
# ============================================================

audio = dataset["train"][0]["audio"]

print("Sampling Rate :", audio["sampling_rate"])
print("Audio Length  :", len(audio["array"]))
print("Duration (sec):",
      round(len(audio["array"]) / audio["sampling_rate"], 2))

print("=" * 70)

# ============================================================
# RANDOM EXAMPLES
# ============================================================

print("Random Spanish Examples")
print("=" * 70)

random.seed(SEED)

indices = random.sample(
    range(len(dataset["train"])),
    5
)

for i, idx in enumerate(indices):

    ex = dataset["train"][idx]

    print(f"\nExample {i+1}")
    print("-" * 60)
    print("ID            :", ex["id"])
    print("Transcription :", ex["transcription"])
    print("Gender        :", ex["gender"])

print("=" * 70)

# ============================================================
# DATASET SUMMARY
# ============================================================

summary = pd.DataFrame({

    "Split": ["Train", "Validation", "Test"],

    "Samples": [

        len(dataset["train"]),
        len(dataset["validation"]),
        len(dataset["test"])

    ]

})

display(summary)

print("=" * 70)
print("FLEURS Spanish Dataset Ready")
print("=" * 70)

LOADING FLEURS SPANISH DATASET
DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 2796
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 408
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 908
    })
})

DATASET INFORMATION
Train Samples      : 2796
Validation Samples : 408
Test Samples       : 908

Dataset Features:
['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id']

VERIFY SPANISH DATASET
Audio Path:
/root/.cache/huggingface/datasets/downloads/extracted/d93a781487030a4952eabce543

,Split,Samples
0,Train,2796
1,Validation,408
2,Test,908


FLEURS Spanish Dataset Ready


In [46]:
# ============================================================
# PART 3 : SPANISH TEXT NORMALIZATION
# ============================================================

print("=" * 70)
print("SPANISH TEXT NORMALIZATION")
print("=" * 70)

import re
import gc

# ============================================================
# NORMALIZATION FUNCTION
# ============================================================

def normalize_text(text):

    if text is None:
        return ""

    text = text.lower().strip()

    # Preserve Spanish characters
    text = re.sub(
        r"[^a-záéíóúüñ0-9\s]",
        "",
        text
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# ============================================================
# CLEAN DATASET
# ============================================================

def clean_transcription(batch):

    batch["transcription"] = normalize_text(
        batch["transcription"]
    )

    return batch

print("Cleaning Spanish Transcriptions...")

dataset = dataset.map(
    clean_transcription,
    desc="Cleaning Text"
)

print("Completed")

print("=" * 70)

# ============================================================
# VERIFY
# ============================================================

print("Sample Normalized Sentences")
print("=" * 70)

for i in range(5):

    print(f"{i+1}. {dataset['train'][i]['transcription']}")

print("=" * 70)

# ============================================================
# DATASET STATISTICS
# ============================================================

all_text = " ".join(dataset["train"]["transcription"])

characters = sorted(list(set(all_text)))

print("Unique Characters :", len(characters))
print(characters)

print("=" * 70)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Text Normalization Complete")
print("=" * 70)

SPANISH TEXT NORMALIZATION
Cleaning Spanish Transcriptions...
Completed
Sample Normalized Sentences
1. los murales o garabatos indeseados reciben el nombre de grafiti
2. por el momento no se sabe qué cargos se imputarán o qué condujo a las autoridades hasta el niño pero se ha dado inicio a procedimientos penales de menores en el tribunal federal
3. si deseas aprender cómo lanzar un búmeran y que vuelva a su mano asegúrate de contar con uno que sea adecuado para el regreso
4. a veces un mismo vuelo puede costar precios muy diferentes en varios recopiladores de contenidos por lo que vale la pena comparar los resultados de búsqueda y navegar en el sitio web de la empresa antes de realizar la reserva
5. los agentes de viajes suelen tener acuerdos con hoteles específicos aunque a través de ellos también es posible reservar otro tipo de alojamiento como zonas de campamento
Unique Characters : 44
[' ', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', '

In [47]:
# ============================================================
# PART 4 : CREATE SPANISH VOCABULARY
# Compatible with Transformers 5.x
# ============================================================

import json

print("=" * 70)
print("CREATING SPANISH VOCABULARY")
print("=" * 70)

# ============================================================
# EXTRACT ALL CHARACTERS
# ============================================================

def extract_all_chars(batch):

    all_text = " ".join(batch["transcription"])

    return {
        "vocab": [list(set(all_text))]
    }

# ============================================================
# TRAIN
# ============================================================

vocab_train = dataset["train"].map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    remove_columns=dataset["train"].column_names,
)

# ============================================================
# VALIDATION
# ============================================================

vocab_valid = dataset["validation"].map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    remove_columns=dataset["validation"].column_names,
)

# ============================================================
# TEST
# ============================================================

vocab_test = dataset["test"].map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    remove_columns=dataset["test"].column_names,
)

# ============================================================
# BUILD VOCAB
# ============================================================

vocab_list = sorted(
    list(
        set(vocab_train["vocab"][0])
        | set(vocab_valid["vocab"][0])
        | set(vocab_test["vocab"][0])
    )
)

# Create dictionary
vocab_dict = {char: idx for idx, char in enumerate(vocab_list)}

# Replace space with '|'
if " " in vocab_dict:
    space_index = vocab_dict[" "]
    del vocab_dict[" "]
    vocab_dict["|"] = space_index
else:
    vocab_dict["|"] = len(vocab_dict)

# Add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocabulary Size :", len(vocab_dict))

# ============================================================
# SAVE VOCAB
# ============================================================

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(
        vocab_dict,
        f,
        ensure_ascii=False,
        indent=4,
        sort_keys=False
    )

print("vocab.json saved successfully.")

print("=" * 70)

# ============================================================
# VERIFY
# ============================================================

print("First 20 Tokens")

for token, idx in list(vocab_dict.items())[:20]:
    print(f"{repr(token):8} -> {idx}")

print("=" * 70)

print("Special Tokens")
print("[UNK] :", vocab_dict["[UNK]"])
print("[PAD] :", vocab_dict["[PAD]"])
print("|     :", vocab_dict["|"])

print("=" * 70)

CREATING SPANISH VOCABULARY
Vocabulary Size : 46
vocab.json saved successfully.
First 20 Tokens
'0'      -> 1
'1'      -> 2
'2'      -> 3
'3'      -> 4
'4'      -> 5
'5'      -> 6
'6'      -> 7
'7'      -> 8
'8'      -> 9
'9'      -> 10
'a'      -> 11
'b'      -> 12
'c'      -> 13
'd'      -> 14
'e'      -> 15
'f'      -> 16
'g'      -> 17
'h'      -> 18
'i'      -> 19
'j'      -> 20
Special Tokens
[UNK] : 44
[PAD] : 45
|     : 0


In [48]:
# ============================================================
# PART 5 : CREATE TOKENIZER + FEATURE EXTRACTOR + PROCESSOR
# Compatible with Transformers 5.13.1
# ============================================================

import gc
import torch

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

print("=" * 70)
print("CREATING TOKENIZER + FEATURE EXTRACTOR + PROCESSOR")
print("=" * 70)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = Wav2Vec2CTCTokenizer(

    "./vocab.json",

    unk_token="[UNK]",

    pad_token="[PAD]",

    word_delimiter_token="|",

    do_lower_case=False

)

print("Tokenizer Created")

# ============================================================
# FEATURE EXTRACTOR
# ============================================================

feature_extractor = Wav2Vec2FeatureExtractor(

    feature_size=1,

    sampling_rate=16000,

    padding_value=0.0,

    do_normalize=True,

    return_attention_mask=True

)

print("Feature Extractor Created")

# ============================================================
# PROCESSOR
# ============================================================

processor = Wav2Vec2Processor(

    feature_extractor=feature_extractor,

    tokenizer=tokenizer

)

print("Processor Created Successfully")

print("=" * 70)

# ============================================================
# SAVE
# ============================================================

processor.save_pretrained("./processor")

print("Processor Saved Successfully")

print("=" * 70)

# ============================================================
# VERIFY
# ============================================================

print("Tokenizer Length      :", len(processor.tokenizer))
print("Vocabulary Size       :", processor.tokenizer.vocab_size)

print("PAD Token             :", processor.tokenizer.pad_token)
print("PAD Token ID          :", processor.tokenizer.pad_token_id)

print("UNK Token             :", processor.tokenizer.unk_token)
print("UNK Token ID          :", processor.tokenizer.unk_token_id)

print("Word Delimiter Token  :", processor.tokenizer.word_delimiter_token)
print("Word Delimiter ID     :", processor.tokenizer.word_delimiter_token_id)

print("=" * 70)

# ============================================================
# TOKENIZATION TEST
# ============================================================

sample_text = dataset["train"][0]["transcription"]

print("Original Text")
print(sample_text)

print("=" * 70)

encoded = processor.tokenizer(

    sample_text,

    add_special_tokens=False

)

print("Input IDs")
print(encoded.input_ids)

print("=" * 70)

decoded = processor.tokenizer.decode(

    encoded.input_ids,

    group_tokens=False

)

print("Decoded Text")
print(decoded)

print("=" * 70)

# ============================================================
# CHECK TOKENIZER CONSISTENCY
# ============================================================

assert decoded == sample_text, \
    "Tokenizer decode does not match original text."

print("Tokenizer Verification Passed")

print("=" * 70)

# ============================================================
# AUDIO TEST
# ============================================================

audio = dataset["train"][0]["audio"]["array"]

inputs = processor(

    audio,

    sampling_rate=16000,

    return_tensors="pt"

)

print("Input Shape :", inputs.input_values.shape)

print("=" * 70)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("PROCESSOR READY")

print("=" * 70)

CREATING TOKENIZER + FEATURE EXTRACTOR + PROCESSOR
Tokenizer Created
Feature Extractor Created
Processor Created Successfully
Processor Saved Successfully
Tokenizer Length      : 48
Vocabulary Size       : 46
PAD Token             : [PAD]
PAD Token ID          : 45
UNK Token             : [UNK]
UNK Token ID          : 44
Word Delimiter Token  : |
Word Delimiter ID     : 0
Original Text
los murales o garabatos indeseados reciben el nombre de grafiti
Input IDs
[22, 25, 29, 0, 23, 31, 28, 11, 22, 15, 29, 0, 25, 0, 17, 11, 28, 11, 12, 11, 30, 25, 29, 0, 19, 24, 14, 15, 29, 15, 11, 14, 25, 29, 0, 28, 15, 13, 19, 12, 15, 24, 0, 15, 22, 0, 24, 25, 23, 12, 28, 15, 0, 14, 15, 0, 17, 28, 11, 16, 19, 30, 19]
Decoded Text
los murales o garabatos indeseados reciben el nombre de grafiti
Tokenizer Verification Passed
Input Shape : torch.Size([1, 92160])
PROCESSOR READY


In [49]:
# ============================================================
# PART 6 : DATA PREPROCESSING
# Compatible with Transformers 5.13.1
# ============================================================

import gc
import numpy as np
import torch

print("=" * 70)
print("PREPARING DATASET")
print("=" * 70)

bad_files = []

# ============================================================
# PREPARE DATASET
# ============================================================

def prepare_dataset(batch):

    try:

        audio = batch["audio"]

        speech = np.nan_to_num(
            audio["array"]
        ).astype(np.float32)

        # ------------------------------------------------
        # INPUT FEATURES
        # ------------------------------------------------

        batch["input_values"] = processor(
            speech,
            sampling_rate=16000
        ).input_values[0]

        batch["input_length"] = len(
            batch["input_values"]
        )

        # ------------------------------------------------
        # LABELS
        # ------------------------------------------------

        batch["labels"] = processor.tokenizer(
            batch["transcription"],
            add_special_tokens=False
        ).input_ids

        return batch

    except Exception as e:

        bad_files.append(str(e))

        return None


# ============================================================
# PREPARE TRAIN
# ============================================================

train_dataset = dataset["train"].map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1,
    desc="Preparing Train"
)

# ============================================================
# PREPARE VALIDATION
# ============================================================

validation_dataset = dataset["validation"].map(
    prepare_dataset,
    remove_columns=dataset["validation"].column_names,
    num_proc=1,
    desc="Preparing Validation"
)

# ============================================================
# PREPARE TEST
# ============================================================

test_dataset = dataset["test"].map(
    prepare_dataset,
    remove_columns=dataset["test"].column_names,
    num_proc=1,
    desc="Preparing Test"
)

print("=" * 70)
print("DATASET PREPARED")
print("=" * 70)

print(train_dataset)
print(validation_dataset)
print(test_dataset)

# ============================================================
# VERIFY
# ============================================================

sample = train_dataset[0]

print("Input Length :", len(sample["input_values"]))
print("Label Length :", len(sample["labels"]))

print("\nDecoded Label:")

print(
    processor.tokenizer.decode(
        sample["labels"],
        group_tokens=False
    )
)

print("=" * 70)

print("Dataset Features")
print(train_dataset.features)

print("=" * 70)

print("Bad Files :", len(bad_files))

if len(bad_files) > 0:
    print(bad_files[:5])

print("=" * 70)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("DATASET READY")
print("=" * 70)

PREPARING DATASET
DATASET PREPARED
Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 2796
})
Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 408
})
Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 908
})
Input Length : 92160
Label Length : 63

Decoded Label:
los murales o garabatos indeseados reciben el nombre de grafiti
Dataset Features
{'input_values': List(Value('float32')), 'input_length': Value('int64'), 'labels': List(Value('int64'))}
Bad Files : 0
DATASET READY


In [50]:
# ============================================================
# PART 7 : DATA COLLATOR
# Compatible with Transformers 5.13.1
# ============================================================

from dataclasses import dataclass
from typing import Dict, List, Union
import torch
import gc

print("=" * 70)
print("CREATING DATA COLLATOR")
print("=" * 70)

# ============================================================
# DATA COLLATOR
# ============================================================

@dataclass
class DataCollatorCTCWithPadding:

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict]
    ) -> Dict[str, torch.Tensor]:

        # ------------------------------------------------
        # INPUT FEATURES
        # ------------------------------------------------

        input_features = [
            {
                "input_values": feature["input_values"]
            }
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # ------------------------------------------------
        # LABEL FEATURES
        # ------------------------------------------------

        label_features = [
            {
                "input_ids": feature["labels"]
            }
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


# ============================================================
# CREATE COLLATOR
# ============================================================

data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("Data Collator Created Successfully")

print("=" * 70)

# ============================================================
# TEST
# ============================================================

sample_batch = [
    train_dataset[0],
    train_dataset[1]
]

batch = data_collator(sample_batch)

print("Input Shape  :", batch["input_values"].shape)
print("Labels Shape :", batch["labels"].shape)

print()

print("Input dtype  :", batch["input_values"].dtype)
print("Labels dtype :", batch["labels"].dtype)

print()

print("First Labels")

print(batch["labels"][0])

print("=" * 70)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("DATA COLLATOR READY")

print("=" * 70)

CREATING DATA COLLATOR
Data Collator Created Successfully
Input Shape  : torch.Size([2, 192960])
Labels Shape : torch.Size([2, 177])

Input dtype  : torch.float32
Labels dtype : torch.int64

First Labels
tensor([  22,   25,   29,    0,   23,   31,   28,   11,   22,   15,   29,    0,
          25,    0,   17,   11,   28,   11,   12,   11,   30,   25,   29,    0,
          19,   24,   14,   15,   29,   15,   11,   14,   25,   29,    0,   28,
          15,   13,   19,   12,   15,   24,    0,   15,   22,    0,   24,   25,
          23,   12,   28,   15,    0,   14,   15,    0,   17,   28,   11,   16,
          19,   30,   19, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -1

In [51]:
# ============================================================
# PART 8 : LOAD WAV2VEC2-XLS-R MODEL
# Compatible with Transformers 5.13.1
# ============================================================

import gc
import torch
from transformers import AutoModelForCTC

print("=" * 70)
print("LOADING WAV2VEC2-XLS-R MODEL")
print("=" * 70)

model = AutoModelForCTC.from_pretrained(

    MODEL_NAME,

    vocab_size=len(processor.tokenizer),

    pad_token_id=processor.tokenizer.pad_token_id,

    bos_token_id=None,

    eos_token_id=None,

    ctc_loss_reduction="mean",

    ctc_zero_infinity=True,

    ignore_mismatched_sizes=True

)

print("Model Loaded Successfully")

print("=" * 70)

# ============================================================
# FREEZE FEATURE ENCODER
# ============================================================

# Recommended for small datasets like FLEURS
model.freeze_feature_encoder()

print("Feature Encoder Frozen")

print("=" * 70)

# ============================================================
# ENABLE GRADIENT CHECKPOINTING
# ============================================================

model.gradient_checkpointing_enable()

print("Gradient Checkpointing Enabled")

print("=" * 70)

# ============================================================
# MOVE MODEL TO DEVICE
# ============================================================

model.to(device)

print("Device :", device)

print("=" * 70)

# ============================================================
# VERIFY MODEL CONFIG
# ============================================================

print("Model Vocabulary :", model.config.vocab_size)
print("Tokenizer Length :", len(processor.tokenizer))

assert model.config.vocab_size == len(processor.tokenizer)

print("Vocabulary Verified")

print("=" * 70)

# ============================================================
# PARAMETERS
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

print("=" * 70)

# ============================================================
# VERIFY LM HEAD
# ============================================================

print("LM Head Output :", model.lm_head.out_features)

assert model.lm_head.out_features == len(processor.tokenizer)

print("LM Head Verified")

print("=" * 70)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("MODEL READY")

print("=" * 70)

LOADING WAV2VEC2-XLS-R MODEL


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded Successfully
Feature Encoder Frozen
Gradient Checkpointing Enabled
Device : cuda
Model Vocabulary : 48
Tokenizer Length : 48
Vocabulary Verified
Total Parameters     : 315,487,920
Trainable Parameters : 311,277,744
LM Head Output : 48
LM Head Verified
MODEL READY


In [52]:
# ============================================================
# PART 9 : COMPUTE METRICS
# Compatible with Transformers 5.13.1
# ============================================================

import evaluate
import numpy as np

print("=" * 70)
print("CREATING METRICS")
print("=" * 70)

# ============================================================
# LOAD METRICS
# ============================================================

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# ============================================================
# COMPUTE METRICS
# ============================================================

def compute_metrics(pred):

    # -----------------------------
    # Predictions
    # -----------------------------

    pred_ids = np.argmax(pred.predictions, axis=-1)

    # -----------------------------
    # Labels
    # -----------------------------

    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # -----------------------------
    # Decode
    # -----------------------------

    pred_str = processor.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False,
        skip_special_tokens=True
    )

    # -----------------------------
    # Normalize whitespace
    # -----------------------------

    pred_str = [
        " ".join(text.split())
        for text in pred_str
    ]

    label_str = [
        " ".join(text.split())
        for text in label_str
    ]

    # -----------------------------
    # Compute Metrics
    # -----------------------------

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": float(wer),
        "cer": float(cer)
    }

print("Metrics Ready")

print("=" * 70)

# ============================================================
# VERIFY
# ============================================================

print("WER Metric :", wer_metric)
print("CER Metric :", cer_metric)

print("=" * 70)

CREATING METRICS
Metrics Ready
WER Metric : EvaluationModule(name: "wer", module_type: "metric", features: {'predictions': Value('string'), 'references': Value('string')}, usage: """
Compute WER score of transcribed segments against references.

Args:
    references: List of references for each speech input.
    predictions: List of transcriptions to score.
    concatenate_texts (bool, default=False): Whether to concatenate all input texts or compute WER iteratively.

Returns:
    (float): the word error rate

Examples:

    >>> predictions = ["this is the prediction", "there is an other sample"]
    >>> references = ["this is the reference", "there is another one"]
    >>> wer = evaluate.load("wer")
    >>> wer_score = wer.compute(predictions=predictions, references=references)
    >>> print(wer_score)
    0.5
""", stored examples: 0)
CER Metric : EvaluationModule(name: "cer", module_type: "metric", features: {'predictions': Value('string'), 'references': Value('string')}, usage: """


In [53]:
# ============================================================
# PART 10 : TRAINING ARGUMENTS
# Compatible with Transformers 5.13.1
# ============================================================

from transformers import TrainingArguments
import torch

print("=" * 70)
print("CREATING TRAINING ARGUMENTS")
print("=" * 70)

training_args = TrainingArguments(

    # =====================================================
    # OUTPUT
    # =====================================================

    output_dir="./wav2vec2-xls-r-spanish",

    # =====================================================
    # TRAINING
    # =====================================================

    do_train=True,
    do_eval=True,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    learning_rate=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

    lr_scheduler_type="linear",

    max_grad_norm=1.0,

    # =====================================================
    # EVALUATION
    # =====================================================

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=25,

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="wer",

    greater_is_better=False,

    # =====================================================
    # PERFORMANCE
    # =====================================================

    fp16=torch.cuda.is_available(),

    gradient_checkpointing=True,

    dataloader_num_workers=2,

    dataloader_pin_memory=True,

    remove_unused_columns=False,

    # =====================================================
    # REPORTING
    # =====================================================

    report_to="none",

    # =====================================================
    # REPRODUCIBILITY
    # =====================================================

    seed=SEED,

)

print(training_args)

print("=" * 70)
print("TrainingArguments Ready")
print("=" * 70)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


CREATING TRAINING ARGUMENTS
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=True,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=I

In [54]:
# ============================================================
# PART 11 : CREATE TRAINER
# Compatible with Transformers 5.13.1
# ============================================================

import gc
import torch
from transformers import Trainer

print("=" * 70)
print("CREATING TRAINER")
print("=" * 70)

trainer = Trainer(

    # ------------------------------------------------
    # MODEL
    # ------------------------------------------------

    model=model,

    # ------------------------------------------------
    # ARGUMENTS
    # ------------------------------------------------

    args=training_args,

    # ------------------------------------------------
    # DATASETS
    # ------------------------------------------------

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    # ------------------------------------------------
    # PROCESSOR
    # ------------------------------------------------

    processing_class=processor,

    # ------------------------------------------------
    # DATA COLLATOR
    # ------------------------------------------------

    data_collator=data_collator,

    # ------------------------------------------------
    # METRICS
    # ------------------------------------------------

    compute_metrics=compute_metrics,

)

print("Trainer Created Successfully")

print("=" * 70)

print("Training Samples   :", len(train_dataset))
print("Validation Samples :", len(validation_dataset))
print("Test Samples       :", len(test_dataset))

print()

print("Tokenizer Size :", len(processor.tokenizer))
print("Model Vocab    :", model.config.vocab_size)

assert len(processor.tokenizer) == model.config.vocab_size

print("Vocabulary Verified")

print("=" * 70)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("TRAINER READY")

print("=" * 70)

CREATING TRAINER
Trainer Created Successfully
Training Samples   : 2796
Validation Samples : 408
Test Samples       : 908

Tokenizer Size : 48
Model Vocab    : 48
Vocabulary Verified
TRAINER READY


In [55]:
# ============================================================
# PART 12 : TRAIN MODEL
# Compatible with Transformers 5.13.1
# ============================================================

import gc
import time
import torch

print("=" * 70)
print("TRAINING START")
print("=" * 70)

start_time = time.time()

# ============================================================
# TRAIN
# ============================================================

train_result = trainer.train()

# ============================================================
# SAVE MODEL
# ============================================================

trainer.save_model("./best_model")
processor.save_pretrained("./best_model")

print("Model Saved Successfully")

print("=" * 70)

# ============================================================
# SAVE STATE
# ============================================================

trainer.save_state()

trainer.save_metrics(
    "train",
    train_result.metrics
)

print("Training State Saved")

print("=" * 70)

# ============================================================
# TRAINING TIME
# ============================================================

elapsed = time.time() - start_time

h = int(elapsed // 3600)
m = int((elapsed % 3600) // 60)
s = int(elapsed % 60)

print(f"Training Time : {h:02d}:{m:02d}:{s:02d}")

print("=" * 70)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("TRAINING FINISHED")

print("=" * 70)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 47, 'bos_token_id': 46}.


TRAINING START


Epoch,Training Loss,Validation Loss,Wer,Cer
1,25.578149,8.293757,1.000000,1.000000
2,12.011644,5.877003,1.000000,1.000000
3,11.653007,5.808049,1.000000,1.000000
4,11.671837,5.791661,1.000000,1.000000
5,11.610249,5.768846,1.000000,1.000000
6,10.123161,3.868027,0.999703,0.581817
7,6.034509,1.504670,0.574270,0.153512
8,4.495522,0.871627,0.358337,0.089871
9,4.001675,0.617755,0.263632,0.066231
10,3.429941,0.532002,0.232756,0.059959


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully
Training State Saved
Training Time : 07:21:47
TRAINING FINISHED
